# Split Strategy

Create leakage-aware train, validation, and test splits from the validated ORIGA segmentation manifest.

The first-pass split will use ORIGA file IDs as split groups so each image-mask pair stays together and no duplicate record can cross split boundaries.

In [3]:
from pathlib import Path
import sys
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
def find_project_root(start_path: Path | None = None) -> Path:
    start_path = start_path or Path.cwd()
    for path in [start_path, *start_path.parents]:
        if (path / "configs").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root containing configs/ and src/")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

manifest_path = PROJECT_ROOT / "data" / "processed" / "manifests" / "origa_segmentation_manifest.csv"

origa_manifest = pd.read_csv(
    manifest_path,
    dtype={
        "file_id": str,
        "split_group_id": str,
    }
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Manifest path: {manifest_path}")
print(f"Manifest rows: {len(origa_manifest)}")

origa_manifest.head()

Project root: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy
Manifest path: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/data/processed/manifests/origa_segmentation_manifest.csv
Manifest rows: 650


,dataset_key,source_dataset,file_id,image_path,mask_path,image_relative_path,mask_relative_path,image_file_name,mask_file_name,mask_encoding,background_value,structure_value_1,structure_value_2,split_group_id,include_in_first_pass,qa_status,notes
0,glaucoma_fundus_imaging_bundle,ORIGA,001,data/external/kaggle/glaucoma_fundus_imaging_b...,data/external/kaggle/glaucoma_fundus_imaging_b...,ORIGA/Images/001.jpg,ORIGA/Masks/001.png,001.jpg,001.png,multiclass_values_0_1_2,0,1,2,ORIGA_001,True,paired_and_visual_sample_reviewed,ORIGA image and mask IDs matched exactly for 6...
1,glaucoma_fundus_imaging_bundle,ORIGA,002,data/external/kaggle/glaucoma_fundus_imaging_b...,data/external/kaggle/glaucoma_fundus_imaging_b...,ORIGA/Images/002.jpg,ORIGA/Masks/002.png,002.jpg,002.png,multiclass_values_0_1_2,0,1,2,ORIGA_002,True,paired_and_visual_sample_reviewed,ORIGA image and mask IDs matched exactly for 6...
2,glaucoma_fundus_imaging_bundle,ORIGA,003,data/external/kaggle/glaucoma_fundus_imaging_b...,data/external/kaggle/glaucoma_fundus_imaging_b...,ORIGA/Images/003.jpg,ORIGA/Masks/003.png,003.jpg,003.png,multiclass_values_0_1_2,0,1,2,ORIGA_003,True,paired_and_visual_sample_reviewed,ORIGA image and mask IDs matched exactly for 6...
3,glaucoma_fundus_imaging_bundle,ORIGA,004,data/external/kaggle/glaucoma_fundus_imaging_b...,data/external/kaggle/glaucoma_fundus_imaging_b...,ORIGA/Images/004.jpg,ORIGA/Masks/004.png,004.jpg,004.png,multiclass_values_0_1_2,0,1,2,ORIGA_004,True,paired_and_visual_sample_reviewed,ORIGA image and mask IDs matched exactly for 6...
4,glaucoma_fundus_imaging_bundle,ORIGA,005,data/external/kaggle/glaucoma_fundus_imaging_b...,data/external/kaggle/glaucoma_fundus_imaging_b...,ORIGA/Images/005.jpg,ORIGA/Masks/005.png,005.jpg,005.png,multiclass_values_0_1_2,0,1,2,ORIGA_005,True,paired_and_visual_sample_reviewed,ORIGA image and mask IDs matched exactly for 6...


In [4]:
RANDOM_SEED = 42

split_dir = PROJECT_ROOT / "data" / "processed" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

groups = (
    origa_manifest[["split_group_id"]]
    .drop_duplicates()
    .sort_values("split_group_id")
    .reset_index(drop=True)
)

train_groups, temp_groups = train_test_split(
    groups,
    test_size=0.30,
    random_state=RANDOM_SEED,
    shuffle=True,
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=RANDOM_SEED,
    shuffle=True,
)

train_groups = train_groups.assign(split="train")
val_groups = val_groups.assign(split="val")
test_groups = test_groups.assign(split="test")

split_assignments = (
    pd.concat([train_groups, val_groups, test_groups], ignore_index=True)
    .sort_values("split_group_id")
    .reset_index(drop=True)
)

origa_manifest_with_splits = origa_manifest.merge(
    split_assignments,
    on="split_group_id",
    how="left",
)

split_summary = (
    origa_manifest_with_splits
    .groupby("split")
    .size()
    .reset_index(name="n_records")
    .sort_values("split")
)

assignments_path = split_dir / "origa_split_assignments.csv"
manifest_with_splits_path = PROJECT_ROOT / "data" / "processed" / "manifests" / "origa_segmentation_manifest_with_splits.csv"

split_assignments.to_csv(assignments_path, index=False)
origa_manifest_with_splits.to_csv(manifest_with_splits_path, index=False)

print(f"Saved split assignments to: {assignments_path}")
print(f"Saved manifest with splits to: {manifest_with_splits_path}")

split_summary

Saved split assignments to: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/data/processed/splits/origa_split_assignments.csv
Saved manifest with splits to: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/data/processed/manifests/origa_segmentation_manifest_with_splits.csv


,split,n_records
0,test,98
1,train,455
2,val,97


In [5]:
expected_total = len(origa_manifest)

actual_total = len(origa_manifest_with_splits)
missing_split_count = origa_manifest_with_splits["split"].isna().sum()

duplicate_group_split_check = (
    origa_manifest_with_splits
    .groupby("split_group_id")["split"]
    .nunique()
    .reset_index(name="n_splits_per_group")
)

groups_crossing_splits = duplicate_group_split_check[
    duplicate_group_split_check["n_splits_per_group"] > 1
]

split_validation = pd.DataFrame(
    [
        {
            "check": "manifest_row_count_preserved",
            "passed": actual_total == expected_total,
            "value": actual_total,
            "expected": expected_total,
        },
        {
            "check": "no_missing_split_assignments",
            "passed": missing_split_count == 0,
            "value": missing_split_count,
            "expected": 0,
        },
        {
            "check": "no_split_group_crosses_splits",
            "passed": len(groups_crossing_splits) == 0,
            "value": len(groups_crossing_splits),
            "expected": 0,
        },
    ]
)

validation_path = split_dir / "origa_split_validation_summary.csv"
split_validation.to_csv(validation_path, index=False)

print(f"Saved split validation summary to: {validation_path}")
split_validation

Saved split validation summary to: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/data/processed/splits/origa_split_validation_summary.csv


,check,passed,value,expected
0,manifest_row_count_preserved,True,650,650
1,no_missing_split_assignments,True,0,0
2,no_split_group_crosses_splits,True,0,0
